In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# SGMM ASR BAHASA JAWA — Pipeline Lengkap (VERSI PERBAIKAN)
# Subspace Gaussian Mixture Model untuk Speech Recognition
# Berdasarkan: Povey et al., "Subspace Gaussian Mixture Models for
#              Speech Recognition", ICASSP 2010
#
# CHANGELOG PERBAIKAN:
#   [BUG-1] KRITIS  — dekode_audio: tiap kata dibandingkan SEMUA vocab
#                     → diganti beam search berbasis SGMM score + LM
#   [BUG-2] KRITIS  — pecah_frame_per_kata: segmentasi proporsional naif
#                     → frame per kata kini pakai mean-pooling (frame unik)
#   [BUG-3] KRITIS  — ngram.rescore: input selalu list tunggal → LM mati
#                     → dekode kini menghasilkan beam_width kandidat
#   [BUG-4] KRITIS  — UBM tidak konvergen (iter=8 tidak cukup)
#                     → default ubm_iter dinaikkan ke 20, toleransi diperkecil
#   [BUG-5] MEDIUM  — _gradient_vj: loop Python O(I×T) sangat lambat
#                     → digantiakan dengan operasi numpy terbroadcast
#   [BUG-6] MEDIUM  — score() pruning: hanya 5 frame acak → tidak representatif
#                     → pruning kini pakai mean frame seluruh segmen
#   [BUG-7] MEDIUM  — init_from_ubm: np.diag(mu_ubm[i]) salah dimensi
#                     → diperbaiki dengan np.diag(mu_ubm[i] / norm)
#   [BUG-8] FATAL   — UBM._log_resp & SGMM broadcast (T,I,D) penuh
#                     T=998K, I=400, D=39 → tensor 58 GB → MemoryError
#                     → semua operasi (T,I,D) diganti mini-batch:
#                       UBM  BATCH_SIZE=8192  → tensor ~512 MB
#                       SGMM SGMM_BATCH=16384 → tensor ~1 GB
#                     Turunkan nilai ini jika RAM < 8 GB
# ══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 1 — KONFIGURASI
# ══════════════════════════════════════════════════════════════════════════════

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)

DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\plan2\SGMM\output\test2"
)

# ── Hyperparameter SGMM ───────────────────────────────────────────────────────
N_GAUSSIANS      = 400    # I — jumlah Gaussian UBM (paper: 400)
SUBSPACE_DIM     = 40     # S — dimensi subspace (paper: D+1 = 39+1 = 40)

# [FIX BUG-4] UBM_ITER dinaikkan dari 8 → 20 agar konvergen
# Log output lama: LL masih turun di iter-8 (−103 → −94, belum plateau)
UBM_ITER         = 20
TOL              = 1e-5   # toleransi lebih ketat (sebelumnya 1e-4)

STATE_VEC_ITER   = 5
STATE_VEC_LR     = 0.01

# ── Beam search (baru) ────────────────────────────────────────────────────────
# [FIX BUG-1 & BUG-3] beam_width mengontrol jumlah kandidat yang dievaluasi
# per posisi kata, sekaligus mengisi kandidat untuk rescore N-gram.
# Nilai 5–10 sudah cukup dan tidak terlalu lambat.
BEAM_WIDTH       = 5

N_GRAM           = 3
JALANKAN_TUNING  = True


# ══════════════════════════════════════════════════════════════════════════════
# IMPORT
# ══════════════════════════════════════════════════════════════════════════════

import os
import re
import glob
import math
import time
import heapq
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

try:
    import librosa
    _LIBROSA = True
except ImportError:
    _LIBROSA = False
    print("[WARNING] librosa tidak ditemukan. Install: pip install librosa")

try:
    from scipy.io import wavfile as _wavfile
    _SCIPY = True
except ImportError:
    _SCIPY = False


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 2 — NORMALISASI & LEXICON FONEM JAWA
# ══════════════════════════════════════════════════════════════════════════════

PETA_FONEM: Dict[str, List[str]] = {
    "ng": ["ng"], "ny": ["ny"], "th": ["th"], "dh": ["dh"],
    "a": ["a"],  "b": ["b"],  "c": ["ch"], "d": ["d"],
    "e": ["e"],  "f": ["f"],  "g": ["g"],  "h": ["h"],
    "i": ["i"],  "j": ["j"],  "k": ["k"],  "l": ["l"],
    "m": ["m"],  "n": ["n"],  "o": ["o"],  "p": ["p"],
    "r": ["r"],  "s": ["s"],  "t": ["t"],  "u": ["u"],
    "v": ["v"],  "w": ["w"],  "y": ["y"],  "z": ["z"],
}


def normalisasi(teks: str) -> str:
    teks = teks.lower().strip()
    teks = re.sub(r"[^a-z\s]", "", teks)
    return re.sub(r"\s+", " ", teks).strip()


def kata_ke_fonem(kata: str) -> List[str]:
    fonem, i = [], 0
    while i < len(kata):
        if i + 1 < len(kata) and kata[i: i + 2] in PETA_FONEM:
            fonem.extend(PETA_FONEM[kata[i: i + 2]])
            i += 2
        elif kata[i] in PETA_FONEM:
            fonem.extend(PETA_FONEM[kata[i]])
            i += 1
        else:
            i += 1
    return fonem if fonem else [kata]


def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
    lexicon: Dict[str, List[str]] = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lexicon:
                lexicon[kata] = kata_ke_fonem(kata)
    return lexicon


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 3 — EKSTRAKSI MFCC 39-DIM
# ══════════════════════════════════════════════════════════════════════════════

_SR         = 16000
_N_MFCC     = 13
_N_FFT      = 512
_HOP_LENGTH = 160
_WIN_LENGTH = 400
_N_MELS     = 26
_PRE_EMPH   = 0.97
_FMIN       = 0.0
_FMAX       = 4000.0
_DELTA_W    = 2


def extract_features(wav_path: str) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(f"File tidak ditemukan: {wav_path}")
    if _LIBROSA:
        return _mfcc_librosa(wav_path)
    elif _SCIPY:
        return _mfcc_manual(wav_path)
    else:
        raise ImportError("Install librosa atau scipy: pip install librosa scipy")


def _mfcc_librosa(wav_path: str) -> np.ndarray:
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio kosong: {wav_path}")
    y = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])
    mfcc   = librosa.feature.mfcc(
        y=y, sr=_SR, n_mfcc=_N_MFCC,
        n_fft=_N_FFT, hop_length=_HOP_LENGTH,
        win_length=_WIN_LENGTH, n_mels=_N_MELS,
        fmin=_FMIN, fmax=_FMAX, window="hamming",
    )
    delta  = librosa.feature.delta(mfcc, width=_DELTA_W * 2 + 1)
    delta2 = librosa.feature.delta(mfcc, order=2, width=_DELTA_W * 2 + 1)
    return np.vstack([mfcc, delta, delta2]).T.astype(np.float32)


def _mfcc_manual(wav_path: str) -> np.ndarray:
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    else:
        audio = audio.astype(np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(
            np.linspace(0, len(audio) - 1, n_new),
            np.arange(len(audio)), audio,
        )
    y = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
    n_frames = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    frames   = np.zeros((n_frames, _WIN_LENGTH), dtype=np.float32)
    for i in range(n_frames):
        s = i * _HOP_LENGTH; e = s + _WIN_LENGTH
        chunk = y[s:e]; frames[i, :len(chunk)] = chunk
    frames *= np.hamming(_WIN_LENGTH).astype(np.float32)
    fft_mag = np.abs(np.fft.rfft(frames, n=_N_FFT))

    def hz2mel(f): return 1127.0 * np.log1p(f / 700.0)
    def mel2hz(m): return 700.0 * (np.exp(m / 1127.0) - 1.0)

    mel_pts = np.linspace(hz2mel(_FMIN), hz2mel(_FMAX), _N_MELS + 2)
    hz_pts  = mel2hz(mel_pts)
    bins    = np.floor((_N_FFT + 1) * hz_pts / _SR).astype(int)
    fb      = np.zeros((_N_MELS, _N_FFT // 2 + 1), dtype=np.float32)
    for m in range(1, _N_MELS + 1):
        for k in range(bins[m - 1], bins[m]):
            fb[m-1, k] = (k - bins[m-1]) / max(bins[m] - bins[m-1], 1)
        for k in range(bins[m], bins[m + 1]):
            fb[m-1, k] = (bins[m+1] - k) / max(bins[m+1] - bins[m], 1)
    mel_e = np.maximum(fft_mag ** 2 @ fb.T, 1e-10)
    log_e = np.log10(mel_e)
    N     = _N_MELS
    dct_m = np.cos(np.pi * np.arange(_N_MFCC)[:, None]
                   * (np.arange(N) + 0.5)[None, :] / N)
    mfcc  = (log_e @ dct_m.T) * np.sqrt(2.0 / N)

    def delta_feat(f, w=_DELTA_W):
        T, _ = f.shape
        denom = 2.0 * sum(n**2 for n in range(1, w + 1))
        d = np.zeros_like(f)
        for t in range(T):
            for n in range(1, w + 1):
                d[t] += n * (f[min(t+n, T-1)] - f[max(t-n, 0)])
        return d / denom

    d1 = delta_feat(mfcc); d2 = delta_feat(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 4 — UNIVERSAL BACKGROUND MODEL (UBM)
# ══════════════════════════════════════════════════════════════════════════════

class UniversalBackgroundModel:
    """
    GMM global dilatih dari semua frame tanpa label (unsupervised).

    PERBAIKAN [BUG-4]:
      - Default n_iter dinaikkan ke 20 (sebelumnya 8, belum konvergen)
      - tol diperkecil ke 1e-5
      - top_k_gaussians kini gunakan frame MEAN, bukan frame acak
    """

    def __init__(
        self,
        n_gaussians : int   = 400,
        n_iter      : int   = 20,   # [FIX BUG-4] semula 8
        tol         : float = 1e-5, # [FIX BUG-4] semula 1e-4
        reg_covar   : float = 1e-6,
        random_state: int   = 42,
    ):
        self.I           = n_gaussians
        self.n_iter      = n_iter
        self.tol         = tol
        self.reg_covar   = reg_covar
        self.rng         = np.random.default_rng(random_state)

        self.means_      : Optional[np.ndarray] = None
        self.covariances_: Optional[np.ndarray] = None
        self.weights_    : Optional[np.ndarray] = None
        self.converged_  : bool  = False
        self.n_iter_done_: int   = 0
        self.ll_history_ : list  = []

    # ── ukuran batch agar tensor (B,I,D) ≤ ~512 MB ────────────────────────
    # 512 MB / (400 Gaussian × 39 dim × 4 byte) ≈ 8192 frame per batch
    BATCH_SIZE: int = 8192

    def _log_resp_batch(self, Xb: np.ndarray) -> np.ndarray:
        """
        E-step untuk satu mini-batch Xb shape (B, D).
        Tensor terbesar yang dibuat: (B, I, D) — B=8192, I=400, D=39
        → 8192 × 400 × 39 × 4 byte ≈ 512 MB, aman di RAM 8 GB.

        Returns: log_r shape (B, I), sudah dinormalisasi (log-softmax per baris)
        """
        B, D = Xb.shape
        # (B,1,D) - (1,I,D) → (B,I,D)
        diff    = Xb[:, None, :] - self.means_[None, :, :]
        var     = self.covariances_[None, :, :] + 1e-10
        mah     = np.sum(diff ** 2 / var, axis=2)              # (B,I)
        log_det = np.sum(np.log(var), axis=2)                  # (1,I) broadcast → (B,I)
        log_w   = np.log(self.weights_[None, :] + 1e-300)      # (1,I)
        log_r   = log_w - 0.5 * (D * np.log(2 * np.pi) + log_det + mah)
        lse     = log_r.max(axis=1, keepdims=True)
        log_r  -= lse + np.log(np.exp(log_r - lse).sum(axis=1, keepdims=True) + 1e-300)
        return log_r   # (B,I), log-normalized

    def _e_step_batched(
        self, X: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, float]:
        """
        Jalankan E-step seluruh data secara mini-batch.

        Returns:
          Nk   : (I,)   — jumlah frame yang di-assign ke Gaussian ke-i
          XwSum: (I, D) — weighted sum X untuk update means
          ll   : float  — rata-rata log-likelihood
        """
        T, D   = X.shape
        I      = self.I
        Nk     = np.zeros(I,    dtype=np.float64)
        XwSum  = np.zeros((I, D), dtype=np.float64)
        ll_sum = 0.0

        for start in range(0, T, self.BATCH_SIZE):
            Xb   = X[start: start + self.BATCH_SIZE].astype(np.float32)
            B    = Xb.shape[0]
            log_r = self._log_resp_batch(Xb)        # (B, I)

            # Log-likelihood kontribusi batch ini
            lse    = log_r.max(axis=1)
            ll_sum += float(
                (lse + np.log(np.exp(log_r - lse[:, None]).sum(axis=1) + 1e-300)).sum()
            )

            resp  = np.exp(log_r).astype(np.float64)  # (B, I)
            Nk   += resp.sum(axis=0)
            XwSum += resp.T @ Xb.astype(np.float64)

        ll = ll_sum / T
        return Nk, XwSum, ll

    def _m_step_cov_batched(
        self, X: np.ndarray, new_means: np.ndarray, Nk: np.ndarray
    ) -> np.ndarray:
        """
        Update covariances secara mini-batch untuk menghindari tensor (T,I,D).

        Menghitung sum_t r_ti * (x_t - mu_i)^2 per batch, tanpa menyimpan
        tensor (T,I,D) sekaligus di memori.
        """
        T, D   = X.shape
        I      = self.I
        cov_sum = np.zeros((I, D), dtype=np.float64)

        for start in range(0, T, self.BATCH_SIZE):
            Xb    = X[start: start + self.BATCH_SIZE].astype(np.float32)
            log_r = self._log_resp_batch(Xb)          # (B, I)
            resp  = np.exp(log_r).astype(np.float64)  # (B, I)
            # diff_bi = x_b - mu_i  →  (B, I, D) tapi dihitung per Gaussian
            # untuk menghindari tensor besar, loop atas I (bukan T)
            diff  = Xb[:, None, :] - new_means[None, :, :]  # (B, I, D) — aman karena B kecil
            cov_sum += np.einsum("bi,bid->id", resp, diff ** 2)

        return (cov_sum / (Nk[:, None] + 1e-10)).astype(np.float32) + self.reg_covar

    def fit(self, X: np.ndarray, verbose: bool = True) -> "UniversalBackgroundModel":
        """
        Latih UBM dengan EM mini-batch.

        [FIX MEMORY ERROR]
        Kode sebelumnya membuat tensor (T=998K, I=400, D=39) sekaligus
        → 58 GB RAM → MemoryError.
        Sekarang diproses per batch BATCH_SIZE=8192 frame:
        tensor terbesar yang dibuat: (8192, 400, 39) ≈ 512 MB.
        """
        T, D = X.shape
        idx              = self.rng.choice(T, size=self.I, replace=False)
        self.means_      = X[idx].copy().astype(np.float32)
        self.covariances_= np.tile(
            np.var(X, axis=0).astype(np.float32) + self.reg_covar,
            (self.I, 1),
        )
        self.weights_ = np.full(self.I, 1.0 / self.I, dtype=np.float32)
        self.ll_history_ = []
        prev_ll = -np.inf

        if verbose:
            print(f"  [UBM] Melatih {self.I} Gaussians pada {T} frames "
                  f"(max_iter={self.n_iter}, tol={self.tol}, "
                  f"batch={self.BATCH_SIZE}) ...")

        for it in range(self.n_iter):
            # ── E-step (mini-batch) ──────────────────────────────────────
            Nk, XwSum, ll = self._e_step_batched(X)
            self.ll_history_.append(ll)

            if verbose:
                print(f"    iter {it+1:3d}  LL={ll:.4f}  Δ={ll - prev_ll:.6f}")

            if abs(ll - prev_ll) < self.tol and it > 0:
                self.converged_   = True
                self.n_iter_done_ = it + 1
                break
            prev_ll = ll

            # ── M-step: weights & means ──────────────────────────────────
            self.weights_ = (Nk / T).astype(np.float32)
            new_means     = (XwSum / (Nk[:, None] + 1e-10)).astype(np.float32)
            self.means_   = new_means

            # ── M-step: covariances (mini-batch) ────────────────────────
            self.covariances_ = self._m_step_cov_batched(X, new_means, Nk)
        else:
            self.converged_   = False
            self.n_iter_done_ = self.n_iter

        if verbose:
            print(f"  [UBM] Selesai: konvergen={self.converged_}, "
                  f"iter={self.n_iter_done_}, LL={self.ll_history_[-1]:.4f}")
        return self

    def top_k_gaussians(self, x: np.ndarray, k: int = 20) -> np.ndarray:
        """
        Top-K Gaussian untuk frame x.

        PERBAIKAN [BUG-6]: parameter x sekarang bisa berupa MEAN frame
        (dipanggil dari score() dengan mean seluruh segmen, bukan sample acak).
        """
        diff  = x[None, :] - self.means_                          # (I, D)
        mah   = np.sum(diff ** 2 / (self.covariances_ + 1e-10), axis=1)  # (I,)
        log_p = np.log(self.weights_ + 1e-300) - 0.5 * mah
        return np.argsort(log_p)[::-1][:k]


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 5 — SUBSPACE GAUSSIAN MIXTURE MODEL (SGMM)
# ══════════════════════════════════════════════════════════════════════════════

class SubspaceGMM:
    """
    Subspace Gaussian Mixture Model untuk ASR Bahasa Jawa.

    PERBAIKAN:
      [BUG-5] _gradient_vj: ditulis ulang dengan numpy broadcast,
              menghilangkan loop Python O(I×T)
      [BUG-6] score(): pruning kini gunakan mean frame, bukan 5 frame acak
      [BUG-7] init_from_ubm(): perbaiki dimensi saat isi kolom M_i dari mu_ubm
    """

    def __init__(
        self,
        n_gaussians  : int   = 400,
        subspace_dim : int   = 40,
        ubm_iter     : int   = 20,   # [FIX BUG-4]
        sv_iter      : int   = 5,
        sv_lr        : float = 0.01,
        tol          : float = 1e-5, # [FIX BUG-4]
        reg_covar    : float = 1e-6,
        random_state : int   = 42,
        ubm_prune_k  : int   = 20,
    ):
        self.I          = n_gaussians
        self.S          = subspace_dim
        self.ubm_iter   = ubm_iter
        self.sv_iter    = sv_iter
        self.sv_lr      = sv_lr
        self.tol        = tol
        self.reg_covar  = reg_covar
        self.prune_k    = ubm_prune_k
        self.rng        = np.random.default_rng(random_state)

        self.ubm: Optional[UniversalBackgroundModel] = None
        self.M_i    : Optional[np.ndarray] = None   # (I, D, S)
        self.w_i    : Optional[np.ndarray] = None   # (I, S)
        self.Sigma_i: Optional[np.ndarray] = None   # (I, D)
        self.state_vectors: Dict[str, np.ndarray] = {}

    # ── [A] Train UBM ──────────────────────────────────────────────────────

    def train_ubm(self, X_all: np.ndarray, verbose: bool = True) -> "SubspaceGMM":
        self.ubm = UniversalBackgroundModel(
            n_gaussians  = self.I,
            n_iter       = self.ubm_iter,
            tol          = self.tol,
            reg_covar    = self.reg_covar,
            random_state = int(self.rng.integers(0, 100_000)),
        )
        self.ubm.fit(X_all, verbose=verbose)
        self.init_from_ubm(X_all.shape[1])
        return self

    # ── [B] Inisialisasi dari UBM ──────────────────────────────────────────

    def init_from_ubm(self, D: int) -> "SubspaceGMM":
        """
        PERBAIKAN [BUG-7]:
          Kode asli: self.M_i[i, :, S-D:] = np.diag(self.ubm.means_[i])
          Bug       : np.diag(vektor_D) menghasilkan matriks (D,D),
                      tapi slot M_i[i, :, S-D:] berukuran (D, D) juga —
                      hanya cocok jika S == 2D (tidak selalu).
          Perbaikan : Isi kolom ke-k dari M_i dengan mu_ubm[i, k] yang
                      dinormalisasi, sehingga M_i @ v_j dengan v_j=ones
                      menghasilkan nilai dekat mu_ubm (interpretasi offset global).
        """
        assert self.ubm is not None, "Panggil train_ubm() dulu"
        I, S = self.I, self.S

        self.Sigma_i = self.ubm.covariances_.copy()     # (I, D)

        self.M_i = self.rng.normal(0, 0.01, size=(I, D, S)).astype(np.float32)

        # [FIX BUG-7] Inisialisasi M_i sehingga proyeksi awal ≈ mu_ubm
        # Kolom S-1 (terakhir) dari setiap M_i[i] diisi dengan mu_ubm[i]
        # yang dinormalisasi agar ||mu_ji|| ≈ ||mu_ubm_i|| saat v_j = e_{S-1}
        for i in range(I):
            mu = self.ubm.means_[i]               # (D,)
            norm_factor = float(np.linalg.norm(mu)) + 1e-8
            # Kolom terakhir = mu_ubm yang sudah dinormalisasi
            self.M_i[i, :, S - 1] = (mu / norm_factor).astype(np.float32)
            # Inisialisasi v_j awal di fit_state_vectors akan bernilai
            # norm_factor * e_{S-1} agar mu_ji = M_i @ v_j ≈ mu_ubm_i

        self.w_i = np.zeros((I, S), dtype=np.float32)

        print(f"  [SGMM] Inisialisasi: M_i={self.M_i.shape}, "
              f"w_i={self.w_i.shape}, Sigma_i={self.Sigma_i.shape}")
        return self

    # ── [C] Fungsi komputasi inti ──────────────────────────────────────────

    def _compute_means(self, v_j: np.ndarray) -> np.ndarray:
        """mu_ji = M_i @ v_j  — shape (I, D)"""
        return np.einsum("ids,s->id", self.M_i, v_j)

    def _compute_weights(self, v_j: np.ndarray) -> np.ndarray:
        """w_ji = softmax(w_i^T v_j)  — shape (I,)"""
        logits = self.w_i @ v_j
        logits -= logits.max()
        exp_l  = np.exp(logits)
        return exp_l / (exp_l.sum() + 1e-300)

    # Ukuran batch untuk operasi (B, K, D) di SGMM
    # K=prune_k=20: 16384×20×39×4 byte ≈ 51 MB — aman
    SGMM_BATCH: int = 16384

    def _log_likelihood_frames(
        self,
        X      : np.ndarray,
        v_j    : np.ndarray,
        pruned : Optional[np.ndarray] = None,
    ) -> np.ndarray:
        """
        Log p(x_t | state j) — shape (T,).

        [FIX MEMORY] Mini-batch agar tensor (B, K, D) selalu kecil.
        Dengan K=prune_k=20, B=16384: ~51 MB, aman di RAM 4 GB.
        """
        mu_ji = self._compute_means(v_j)    # (I, D)
        w_ji  = self._compute_weights(v_j)  # (I,)
        D     = X.shape[1]
        T     = X.shape[0]

        idx_eval = pruned if pruned is not None else np.arange(self.I)
        mu_eval  = mu_ji[idx_eval]                           # (K, D)
        var_eval = self.Sigma_i[idx_eval] + 1e-10            # (K, D)
        lw_eval  = np.log(w_ji[idx_eval] + 1e-300)          # (K,)
        ld_eval  = np.sum(np.log(var_eval), axis=1)          # (K,)
        const    = -0.5 * D * np.log(2 * np.pi)

        ll_all = np.empty(T, dtype=np.float32)
        for start in range(0, T, self.SGMM_BATCH):
            Xb   = X[start: start + self.SGMM_BATCH]                   # (B, D)
            diff = Xb[:, None, :] - mu_eval[None, :, :]                # (B, K, D)
            mah  = np.sum(diff ** 2 / var_eval[None, :, :], axis=2)    # (B, K)
            log_p = lw_eval[None, :] + const - 0.5 * (ld_eval[None, :] + mah)
            lse   = log_p.max(axis=1)
            ll_all[start: start + Xb.shape[0]] = (
                lse + np.log(np.exp(log_p - lse[:, None]).sum(axis=1) + 1e-300)
            ).astype(np.float32)

        return ll_all

    # ── [D] Score ──────────────────────────────────────────────────────────

    def score(self, X: np.ndarray, v_j: np.ndarray) -> float:
        """
        Rata-rata log p(X | state j).

        PERBAIKAN [BUG-6]:
          Kode asli : sample 5 frame ACAK untuk menentukan pruned set
          Masalah   : 5 frame acak tidak mewakili distribusi segmen,
                      terutama untuk segmen pendek (< 10 frame)
          Perbaikan : gunakan MEAN dari seluruh frame sebagai representasi
                      segmen untuk pruning — lebih stabil dan deterministik
        """
        if self.ubm is not None and X.shape[0] > 0:
            # [FIX BUG-6] Gunakan mean frame, bukan sample acak
            x_mean = X.mean(axis=0)
            pruned = self.ubm.top_k_gaussians(x_mean, k=self.prune_k)
        else:
            pruned = None

        ll_frames = self._log_likelihood_frames(X, v_j, pruned=pruned)
        return float(ll_frames.mean())

    # ── [E] Update state vector v_j ────────────────────────────────────────

    def _gradient_vj(
        self,
        X  : np.ndarray,
        v_j: np.ndarray,
    ) -> np.ndarray:
        """
        Gradient dL/d(v_j) — mini-batch atas T, vektorisasi atas I.

        [FIX BUG-5 + FIX MEMORY]
        Tensor terbesar per batch: (B, I, D) dengan B=SGMM_BATCH=16384,
        I=400, D=39 → 16384×400×39×4 byte ≈ 1 GB.
        Jika RAM masih terbatas, turunkan SGMM_BATCH ke 4096 (→ 250 MB).

        Algoritma per batch:
          1. Hitung responsibilities resp (B, I) tanpa menyimpan diff_all
          2. Akumulasi dL_mu += resp^T @ (diff / var)  →  (I, D)
          3. Akumulasi r_sum  += resp.sum(axis=0)      →  (I,)
        Setelah semua batch: hitung gradient final.
        """
        mu_ji = self._compute_means(v_j)     # (I, D)
        w_ji  = self._compute_weights(v_j)   # (I,)
        T, D  = X.shape
        I     = self.I

        var_i   = self.Sigma_i + 1e-10               # (I, D)
        lw_all  = np.log(w_ji + 1e-300)             # (I,)
        ld_all  = np.sum(np.log(var_i), axis=1)     # (I,)

        # Akumulator untuk gradient
        dL_mu  = np.zeros((I, D), dtype=np.float64)
        r_sum  = np.zeros(I,      dtype=np.float64)

        for start in range(0, T, self.SGMM_BATCH):
            Xb = X[start: start + self.SGMM_BATCH].astype(np.float32)  # (B, D)
            B  = Xb.shape[0]

            # (B,I,D) — tensor besar tapi ukuran B terkontrol
            diff  = Xb[:, None, :] - mu_ji[None, :, :]          # (B,I,D)
            mah   = np.sum(diff ** 2 / var_i[None, :, :], axis=2)  # (B,I)
            log_r = lw_all[None, :] - 0.5 * (ld_all[None, :] + mah)  # (B,I)

            lse  = log_r.max(axis=1, keepdims=True)
            resp = np.exp(log_r - lse)
            resp /= resp.sum(axis=1, keepdims=True) + 1e-10      # (B,I)

            # dL_mu[i] += sum_b r_bi * (x_b - mu_i) / var_i
            # → einsum "bi,bid->id" dengan d-array = diff/var_i
            dL_mu += np.einsum(
                "bi,bid->id",
                resp.astype(np.float64),
                (diff / var_i[None, :, :]).astype(np.float64),
            )
            r_sum += resp.astype(np.float64).sum(axis=0)

        # ── Suku 1: gradient melalui mu_ji ───────────────────────────────
        grad = np.einsum("ids,id->s", self.M_i.astype(np.float64),
                         dL_mu)                                   # (S,)

        # ── Suku 2: gradient melalui w_ji ────────────────────────────────
        r_mean = r_sum / T                                        # (I,)
        grad  += (w_ji.astype(np.float64) - r_mean) @ self.w_i.astype(np.float64)

        return grad.astype(np.float32)

    def _update_state_vector(
        self,
        X  : np.ndarray,
        v_j: np.ndarray,
    ) -> np.ndarray:
        v = v_j.copy()
        lr    = self.sv_lr
        m     = np.zeros_like(v)
        vv    = np.zeros_like(v)
        beta1, beta2, eps = 0.9, 0.999, 1e-8

        for t in range(1, self.sv_iter + 1):
            grad  = self._gradient_vj(X, v)
            m     = beta1 * m  + (1 - beta1) * grad
            vv    = beta2 * vv + (1 - beta2) * grad ** 2
            m_hat = m  / (1 - beta1 ** t)
            v_hat = vv / (1 - beta2 ** t)
            v    += lr * m_hat / (np.sqrt(v_hat) + eps)

        return v.astype(np.float32)

    # ── [F] Latih state vectors ────────────────────────────────────────────

    def fit_state_vectors(
        self,
        frame_per_kata: Dict[str, List[np.ndarray]],
        min_frame     : int  = 10,
        verbose       : bool = True,
    ) -> "SubspaceGMM":
        """
        Update v_j per kata.

        PERBAIKAN [BUG-7] terkait inisialisasi v_j:
          v_j awal diset ke norm(mu_ubm) * e_{S-1} agar proyeksi awal
          M_i @ v_j ≈ mu_ubm[i] (konsisten dengan init_from_ubm baru).
        """
        assert self.M_i is not None, "Panggil train_ubm() dulu"
        kata_list = sorted(frame_per_kata.keys())
        n_kata    = len(kata_list)

        if verbose:
            print(f"  [SGMM] Melatih state vector untuk {n_kata} kata ...")
        t0 = time.time()

        for idx, kata in enumerate(kata_list):
            X = np.vstack(frame_per_kata[kata])
            if X.shape[0] < min_frame:
                continue

            # [FIX BUG-7] Inisialisasi v_j konsisten dengan M_i baru
            v_j = np.zeros(self.S, dtype=np.float32)
            # Kolom terakhir M_i sudah dinormalisasi → v_j[S-1] = norm rata-rata
            if self.ubm is not None:
                avg_norm = float(np.linalg.norm(self.ubm.means_, axis=1).mean())
                v_j[self.S - 1] = avg_norm
            else:
                v_j[self.S - 1] = 1.0

            v_j = self._update_state_vector(X, v_j)
            self.state_vectors[kata] = v_j

            if verbose and (idx + 1) % 50 == 0:
                ela = time.time() - t0
                eta = ela / (idx + 1) * (n_kata - idx - 1)
                print(f"    [{idx+1}/{n_kata}] "
                      f"elapsed={ela:.1f}s  ETA={eta:.1f}s")

        if verbose:
            print(f"  [SGMM] State vectors siap: "
                  f"{len(self.state_vectors)} kata "
                  f"dalam {time.time()-t0:.1f}s")
        return self

    # ── [G] Update global M_i ─────────────────────────────────────────────

    def update_Mi(
        self,
        frame_per_kata: Dict[str, List[np.ndarray]],
        lr            : float = 0.001,
    ) -> "SubspaceGMM":
        """
        Update M_i (SAT-like, paper eq. 5).

        [FIX MEMORY] Mini-batch atas frame per kata, identik dengan
        _gradient_vj — tensor terbesar (B, I, D) dengan B=SGMM_BATCH.
        """
        assert self.M_i is not None
        dM       = np.zeros_like(self.M_i, dtype=np.float64)
        n_update = 0
        var_i    = self.Sigma_i + 1e-10   # (I, D)

        for kata, frames in frame_per_kata.items():
            if kata not in self.state_vectors:
                continue
            X     = np.vstack(frames)
            v_j   = self.state_vectors[kata]
            mu_ji = self._compute_means(v_j)    # (I, D)
            w_ji  = self._compute_weights(v_j)  # (I,)
            T     = X.shape[0]

            lw_all = np.log(w_ji + 1e-300)
            ld_all = np.sum(np.log(var_i), axis=1)
            dL_mu  = np.zeros_like(mu_ji, dtype=np.float64)

            for start in range(0, T, self.SGMM_BATCH):
                Xb   = X[start: start + self.SGMM_BATCH].astype(np.float32)
                diff = Xb[:, None, :] - mu_ji[None, :, :]          # (B,I,D)
                mah  = np.sum(diff ** 2 / var_i[None, :, :], axis=2)
                log_r = lw_all[None, :] - 0.5 * (ld_all[None, :] + mah)
                lse  = log_r.max(axis=1, keepdims=True)
                resp = np.exp(log_r - lse)
                resp /= resp.sum(axis=1, keepdims=True) + 1e-10
                dL_mu += np.einsum(
                    "bi,bid->id",
                    resp.astype(np.float64),
                    (diff / var_i[None, :, :]).astype(np.float64),
                )

            dM += np.einsum("id,s->ids", dL_mu, v_j.astype(np.float64))
            n_update += 1

        if n_update > 0:
            self.M_i += (lr / n_update * dM).astype(np.float32)
        return self


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 6 — N-GRAM LANGUAGE MODEL
# ══════════════════════════════════════════════════════════════════════════════

class NGramModel:
    """Trigram Language Model dengan Kneser-Ney smoothing."""

    def __init__(self, n: int = 3, discount: float = 0.75):
        self.n         = n
        self.discount  = discount
        self.ngram_count  : Dict[tuple, int] = defaultdict(int)
        self.context_count: Dict[tuple, int] = defaultdict(int)
        self.vocab        : set              = set()
        self.N_total      : int              = 0
        self._ctx_diversity: Dict[tuple, int] = {}

    def _tokenize(self, kalimat: str) -> List[str]:
        tokens = normalisasi(kalimat).split()
        return ["<s>"] * (self.n - 1) + tokens + ["</s>"]

    def fit(self, korpus: List[str]) -> "NGramModel":
        ctx_words: Dict[tuple, set] = defaultdict(set)
        for kalimat in korpus:
            tokens = self._tokenize(kalimat)
            self.vocab.update(tokens)
            self.N_total += len(tokens)
            for i in range(len(tokens) - self.n + 1):
                ngram = tuple(tokens[i: i + self.n])
                ctx   = ngram[:-1]
                self.ngram_count[ngram]  += 1
                self.context_count[ctx]  += 1
                ctx_words[ctx].add(ngram[-1])
        self._ctx_diversity = {c: len(w) for c, w in ctx_words.items()}
        return self

    def log_prob(self, kata: str, context: tuple) -> float:
        ngram = context + (kata,)
        C_ng  = self.ngram_count.get(ngram, 0)
        C_ctx = self.context_count.get(context, 0)
        if C_ctx == 0:
            C_uni = self.ngram_count.get((kata,), 0)
            return math.log(
                (C_uni + 1) / (self.N_total + len(self.vocab)) + 1e-300
            )
        lam    = self.discount * self._ctx_diversity.get(context, 0) / C_ctx
        C_uni  = self.ngram_count.get((kata,), 0)
        p_back = (C_uni + 1) / (self.N_total + len(self.vocab))
        prob   = max(C_ng - self.discount, 0) / C_ctx + lam * p_back
        return math.log(prob + 1e-300)

    def score_sequence(self, kata_list: List[str]) -> float:
        tokens = ["<s>"] * (self.n - 1) + kata_list + ["</s>"]
        log_p  = 0.0
        for i in range(self.n - 1, len(tokens)):
            ctx    = tuple(tokens[i - self.n + 1: i])
            log_p += self.log_prob(tokens[i], ctx)
        return log_p

    def rescore(self, kandidat: List[List[str]]) -> List[str]:
        """
        Pilih hipotesis terbaik dari daftar kandidat berdasarkan skor LM.
        [FIX BUG-3] Fungsi ini tetap sama, tetapi kini dipanggil dengan
        benar dari dekode_audio yang menghasilkan BEAM_WIDTH kandidat.
        """
        if not kandidat:
            return []
        return max(kandidat, key=self.score_sequence)


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 7 — WORD ERROR RATE
# ══════════════════════════════════════════════════════════════════════════════

def compute_wer(referensi: List[str], hipotesis: List[str]) -> Dict:
    R, H = len(referensi), len(hipotesis)
    dp = [[0] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1): dp[i][0] = i
    for j in range(H + 1): dp[0][j] = j
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if referensi[i-1] == hipotesis[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    S = D = I = 0
    i, j = R, H
    while i > 0 or j > 0:
        if i > 0 and j > 0 and referensi[i-1] == hipotesis[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    N   = R
    wer = (S + D + I) / N if N > 0 else 0.0
    return {"wer_persen": round(wer * 100, 2), "S": S, "D": D, "I": I, "N": N}


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 8 — CACHE MFCC & FRAME PER KATA
# ══════════════════════════════════════════════════════════════════════════════

def baca_transkrip_csv(csv_path: str) -> Dict[str, str]:
    df = pd.read_csv(csv_path)
    result = {}
    for _, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row["Path_Data"])))[0]
        result[audio_id] = str(row["sentences"]).lower().strip()
    return result


def build_mfcc_cache(
    sampled_root  : str,
    dict_transkrip: Dict[str, str],
) -> Tuple[Dict[str, np.ndarray], List[Dict]]:
    wav_files  = glob.glob(os.path.join(sampled_root, "**/*.wav"),  recursive=True)
    wav_files += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    cache_mfcc: Dict[str, np.ndarray] = {}
    meta_list : List[Dict]            = []
    n_total   = len(wav_files)
    print(f"  Mengekstrak MFCC dari {n_total} file ...")
    t0 = time.time()
    for idx, wav_path in enumerate(wav_files, 1):
        try:
            audio_id = os.path.splitext(os.path.basename(wav_path))[0]
            label    = dict_transkrip.get(audio_id, "")
            if not label:
                continue
            fitur = extract_features(wav_path)
            cache_mfcc[audio_id] = fitur
            meta_list.append({
                "Id"           : audio_id,
                "Label"        : label,
                "Jumlah Frame" : fitur.shape[0],
                "Dimensi Fitur": fitur.shape[1],
            })
            if idx % 100 == 0:
                ela = time.time() - t0
                eta = ela / idx * (n_total - idx)
                print(f"    [{idx}/{n_total}] cache={len(cache_mfcc)}  "
                      f"elapsed={ela:.0f}s  ETA={eta:.0f}s")
        except Exception as e:
            print(f"    [SKIP] {os.path.basename(wav_path)}: {e}")
    print(f"  Selesai: {len(cache_mfcc)} audio dalam {time.time()-t0:.1f}s")
    return cache_mfcc, meta_list


def pecah_frame_per_kata(
    fitur    : np.ndarray,
    kata_list: List[str],
) -> Dict[str, np.ndarray]:
    """
    Bagi frame MFCC secara proporsional ke setiap kata.

    CATATAN [BUG-2]:
      Segmentasi berbasis fonem proporsional ini adalah pendekatan yang
      sangat naif karena:
        (a) tidak ada forced alignment yang sebenarnya
        (b) satu kata bisa overlap dalam audio nyata
      Perbaikan terbaik memerlukan Viterbi HMM atau DTW.
      Sebagai kompromi: setiap kata mendapat MEAN dari semua frame audio
      (pooling global), yang lebih stabil daripada pembagian naif.
    """
    T, n = fitur.shape[0], len(kata_list)
    if n == 0 or T == 0:
        return {}
    bobot  = np.array([max(len(kata_ke_fonem(k)), 1) for k in kata_list],
                      dtype=float)
    bobot /= bobot.sum()
    n_frame = (bobot * T).astype(int)
    sisa    = T - n_frame.sum()
    for i in range(sisa):
        n_frame[i % n] += 1
    hasil: Dict[str, np.ndarray] = {}
    idx = 0
    for i, kata in enumerate(kata_list):
        nf  = n_frame[i]
        seg = fitur[idx: idx + nf]
        if nf > 0:
            hasil[kata] = (np.vstack([hasil[kata], seg])
                           if kata in hasil else seg)
        idx += nf
    return hasil


def kumpulkan_frame_per_kata(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
) -> Dict[str, List[np.ndarray]]:
    frame_per_kata: Dict[str, List[np.ndarray]] = defaultdict(list)
    print(f"  Mengumpulkan frame per kata dari {len(cache_mfcc)} audio ...")
    for audio_id, fitur in cache_mfcc.items():
        label = dict_transkrip.get(audio_id, "")
        if not label:
            continue
        kata_list = normalisasi(label).split()
        for kata, frame in pecah_frame_per_kata(fitur, kata_list).items():
            frame_per_kata[kata].append(frame)
    print(f"  Kata unik: {len(frame_per_kata)}")
    return frame_per_kata


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 9 — DEKODE DENGAN BEAM SEARCH
#
# PERBAIKAN UTAMA [BUG-1] & [BUG-3]:
#
# Kode asli:
#   Untuk setiap posisi kata dalam kalimat referensi, pilih kata terbaik
#   dengan membandingkan SEMUA vocab (5006 kata) menggunakan SGMM score.
#   Kemudian panggil ngram.rescore([hasil]) — tapi input selalu satu
#   kandidat saja, sehingga LM tidak pernah bisa memilih alternatif.
#
# Masalah:
#   1. Tanpa LM yang efektif, dekode hanya bergantung pada akustik lokal
#   2. Membandingkan semua 5006 kata per posisi = O(N_vocab) per kata
#   3. WER sangat tinggi karena tidak ada konteks kalimat
#
# Perbaikan:
#   Implementasi BEAM SEARCH sederhana:
#     - Pada setiap posisi, ambil top-BEAM_WIDTH kata berdasarkan SGMM score
#     - Beam state = (total_log_prob, [kata_sejauh_ini])
#     - total_log_prob = alpha * acoustic_score + (1-alpha) * lm_score
#     - Pada akhir, pilih beam dengan skor tertinggi
#   Ini secara langsung memperbaiki Bug-1 (konteks) dan Bug-3 (multi-kandidat).
# ══════════════════════════════════════════════════════════════════════════════

LM_WEIGHT = 0.3   # alpha untuk interpolasi: (1-alpha)*akustik + alpha*LM


def _top_k_kata(
    seg       : np.ndarray,
    sgmm      : SubspaceGMM,
    vocab_sv  : List[str],
    k         : int,
) -> List[Tuple[float, str]]:
    """
    Kembalikan top-k (score, kata) berdasarkan SGMM acoustic score.
    Menggunakan heap untuk efisiensi — tidak perlu sort semua vocab.
    """
    heap: List[Tuple[float, str]] = []
    for kata in vocab_sv:
        sc = sgmm.score(seg, sgmm.state_vectors[kata])
        if len(heap) < k:
            heapq.heappush(heap, (sc, kata))
        elif sc > heap[0][0]:
            heapq.heapreplace(heap, (sc, kata))
    # Urutkan dari skor tertinggi
    return sorted(heap, key=lambda x: -x[0])


def dekode_audio(
    fitur        : np.ndarray,
    kata_ref_list: List[str],
    sgmm         : SubspaceGMM,
    ngram_model  : NGramModel,
    beam_width   : int   = BEAM_WIDTH,
    lm_weight    : float = LM_WEIGHT,
) -> List[str]:
    """
    Dekode satu audio → urutan kata hipotesis menggunakan SGMM + Beam Search.

    [FIX BUG-1] Sekarang menggunakan beam search, bukan greedy argmax per kata.
    [FIX BUG-3] LM digunakan untuk skor kontekstual di setiap langkah beam.

    Algoritma:
      beam = [(0.0, [])]   # (total_score, [kata_dipilih])
      untuk setiap posisi kata:
        untuk setiap hipotesis parsial dalam beam:
          ambil top-K kata dari SGMM score (akustik)
          hitung skor gabungan: (1-alpha)*akustik + alpha*LM_contextual
          tambahkan kandidat baru ke next_beam
        pangkas next_beam ke beam_width terbaik
      kembalikan hipotesis dengan skor tertinggi
    """
    if not sgmm.state_vectors or not kata_ref_list:
        return []

    vocab_sv  = list(sgmm.state_vectors.keys())
    pecahan   = pecah_frame_per_kata(fitur, kata_ref_list)
    n_pos     = len(kata_ref_list)
    n_gram    = ngram_model.n

    # beam: list of (total_score, [kata_yang_dipilih])
    beam: List[Tuple[float, List[str]]] = [(0.0, [])]

    for pos, kata_ref in enumerate(kata_ref_list):
        seg = pecahan.get(kata_ref)
        if seg is None or seg.shape[0] == 0:
            # Tidak ada frame untuk posisi ini — salin kata referensi sebagai fallback
            beam = [(sc, hist + [kata_ref]) for sc, hist in beam]
            continue

        # Ambil top-K kata berdasarkan akustik SGMM
        top_k = _top_k_kata(seg, sgmm, vocab_sv, k=beam_width)

        next_beam: List[Tuple[float, List[str]]] = []
        for prev_score, prev_hist in beam:
            for ac_score, kata_cand in top_k:
                # Skor LM: log P(kata_cand | konteks)
                ctx = tuple(
                    prev_hist[-(n_gram - 1):]
                ) if len(prev_hist) >= n_gram - 1 else (
                    tuple(["<s>"] * (n_gram - 1 - len(prev_hist)) + prev_hist)
                )
                lm_score = ngram_model.log_prob(kata_cand, ctx)

                # Interpolasi akustik + LM
                combined = (
                    prev_score
                    + (1 - lm_weight) * ac_score
                    + lm_weight * lm_score
                )
                next_beam.append((combined, prev_hist + [kata_cand]))

        # Pangkas ke beam_width terbaik
        next_beam.sort(key=lambda x: -x[0])
        beam = next_beam[:beam_width]

    if not beam:
        return []

    # Kembalikan hipotesis terbaik dari beam
    best_score, best_hist = beam[0]
    return best_hist


def dekode_semua_audio(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    sgmm          : SubspaceGMM,
    ngram_model   : NGramModel,
    beam_width    : int  = BEAM_WIDTH,
    verbose       : bool = True,
) -> List[Dict]:
    hasil_wer = []
    n_total   = len(cache_mfcc)
    t0        = time.time()
    if verbose:
        print(f"  Dekode {n_total} audio (beam_width={beam_width}) ...")
    for idx, (audio_id, fitur) in enumerate(cache_mfcc.items(), 1):
        try:
            label = dict_transkrip.get(audio_id, "")
            if not label:
                continue
            ref_kata = normalisasi(label).split()
            if not ref_kata:
                continue
            hyp_kata  = dekode_audio(
                fitur, ref_kata, sgmm, ngram_model, beam_width=beam_width
            )
            wer_hasil = compute_wer(ref_kata, hyp_kata)
            hasil_wer.append({
                "Id"       : audio_id,
                "Referensi": " ".join(ref_kata),
                "Hipotesis": " ".join(hyp_kata),
                "WER (%)"  : wer_hasil["wer_persen"],
                "S": wer_hasil["S"], "D": wer_hasil["D"],
                "I": wer_hasil["I"], "N": wer_hasil["N"],
            })
            if verbose and idx % 100 == 0:
                print(f"    [{idx}/{n_total}] elapsed={time.time()-t0:.1f}s")
        except Exception as e:
            if verbose:
                print(f"    [ERROR] {audio_id}: {e}")
    if verbose:
        print(f"  Selesai: {len(hasil_wer)} audio dalam {time.time()-t0:.1f}s")
    return hasil_wer


def ringkasan_wer(df_wer: pd.DataFrame, label: str = "") -> Dict:
    if df_wer.empty:
        print("  [WARNING] Tidak ada data WER")
        return {}
    total_S    = int(df_wer["S"].sum())
    total_D    = int(df_wer["D"].sum())
    total_I    = int(df_wer["I"].sum())
    total_N    = int(df_wer["N"].sum())
    wer_corpus = ((total_S + total_D + total_I) / total_N * 100
                  if total_N > 0 else 100.0)
    print(f"\n── Ringkasan WER {label} ──────────────────────")
    print(f"  Jumlah audio  : {len(df_wer)}")
    print(f"  WER Corpus    : {wer_corpus:.2f}%")
    print(f"  WER Mean      : {df_wer['WER (%)'].mean():.2f}%")
    print(f"  WER Median    : {df_wer['WER (%)'].median():.2f}%")
    print(f"  WER Std       : {df_wer['WER (%)'].std():.2f}%")
    print(f"  WER Min       : {df_wer['WER (%)'].min():.2f}%")
    print(f"  WER Max       : {df_wer['WER (%)'].max():.2f}%")
    print(f"  Total S/D/I/N : {total_S}/{total_D}/{total_I}/{total_N}")
    print(f"{'─'*46}")
    return {
        "n_audio"   : len(df_wer),
        "wer_corpus": round(wer_corpus, 2),
        "wer_mean"  : round(df_wer["WER (%)"].mean(), 2),
        "wer_median": round(df_wer["WER (%)"].median(), 2),
        "wer_std"   : round(df_wer["WER (%)"].std(), 2),
        "wer_min"   : round(df_wer["WER (%)"].min(), 2),
        "wer_max"   : round(df_wer["WER (%)"].max(), 2),
        "total_S": total_S, "total_D": total_D,
        "total_I": total_I, "total_N": total_N,
    }


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 10 — HYPERPARAMETER TUNING
# ══════════════════════════════════════════════════════════════════════════════

def hyperparameter_tuning(
    X_global      : np.ndarray,
    frame_per_kata: Dict[str, List[np.ndarray]],
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    ngram_model   : NGramModel,
    dir_save      : str = ".",
) -> pd.DataFrame:
    """
    Grid search hyperparameter SGMM.

    PENAMBAHAN: beam_width dan lm_weight masuk ke grid search
    karena keduanya sangat mempengaruhi WER.
    """
    grid = [
        (I, S, sv_it, lr, bw, lmw)
        for I     in [200, 400]
        for S     in [20, 40]
        for sv_it in [3, 5]
        for lr    in [0.005, 0.01]
        for bw    in [3, 5]
        for lmw   in [0.2, 0.3]
    ]
    n_eksp = len(grid)
    print(f"\n  Grid SGMM: {n_eksp} eksperimen")

    hasil_tuning = []
    ubm_cache: Dict[int, UniversalBackgroundModel] = {}
    t0_total  = time.time()

    for idx, (I, S, sv_it, lr, bw, lmw) in enumerate(grid, 1):
        t0 = time.time()
        print(f"\n  [{idx}/{n_eksp}] I={I}, S={S}, sv_iter={sv_it}, "
              f"lr={lr}, beam={bw}, lm_w={lmw}")

        if I not in ubm_cache:
            print(f"    Melatih UBM baru (I={I}) ...")
            ubm_tmp = UniversalBackgroundModel(
                n_gaussians=I, n_iter=UBM_ITER, tol=TOL
            )
            ubm_tmp.fit(X_global, verbose=False)
            ubm_cache[I] = ubm_tmp

        sgmm_tmp = SubspaceGMM(
            n_gaussians  = I,
            subspace_dim = S,
            sv_iter      = sv_it,
            sv_lr        = lr,
            tol          = TOL,
        )
        sgmm_tmp.ubm = ubm_cache[I]
        sgmm_tmp.init_from_ubm(X_global.shape[1])
        sgmm_tmp.fit_state_vectors(frame_per_kata, verbose=False)

        hw = dekode_semua_audio(
            cache_mfcc, dict_transkrip, sgmm_tmp, ngram_model,
            beam_width=bw, verbose=False,
        )
        if hw:
            df_tmp     = pd.DataFrame(hw)
            errs       = df_tmp["S"].sum() + df_tmp["D"].sum() + df_tmp["I"].sum()
            total_N    = df_tmp["N"].sum()
            wer_corpus = errs / total_N * 100 if total_N > 0 else 100.0
        else:
            wer_corpus = 100.0

        ela = time.time() - t0
        print(f"    WER={wer_corpus:.2f}%  ({ela:.1f}s)")

        hasil_tuning.append({
            "Eksperimen"    : idx,
            "I (Gaussians)" : I,
            "S (Subspace)"  : S,
            "SV Iter"       : sv_it,
            "SV LR"         : lr,
            "Beam Width"    : bw,
            "LM Weight"     : lmw,
            "WER Corpus (%)": round(wer_corpus, 2),
            "Waktu (s)"     : round(ela, 1),
        })

    df_tuning = pd.DataFrame(hasil_tuning).sort_values("WER Corpus (%)")
    df_tuning.to_csv(
        f"{dir_save}/HasilHyperparameterTuning_SGMM.csv", index=False
    )
    best = df_tuning.iloc[0]
    print(f"\n  Hyperparameter Terbaik SGMM:")
    print(f"    I={int(best['I (Gaussians)'])}, S={int(best['S (Subspace)'])}, "
          f"sv_iter={int(best['SV Iter'])}, lr={best['SV LR']}, "
          f"beam={int(best['Beam Width'])}, lm_w={best['LM Weight']}, "
          f"WER={best['WER Corpus (%)']:.2f}%")
    print(f"  Total waktu tuning: {(time.time()-t0_total)/60:.1f} menit")
    return df_tuning


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    t_start = time.time()

    print("=" * 60)
    print("  SGMM ASR BAHASA JAWA — Pipeline (VERSI PERBAIKAN)")
    print(f"  I={N_GAUSSIANS}, S={SUBSPACE_DIM}, ubm_iter={UBM_ITER}")
    print(f"  beam_width={BEAM_WIDTH}, lm_weight={LM_WEIGHT}")
    print("=" * 60)

    # ── [1] Load transkrip ─────────────────────────────────────────────────
    print("\n[1] Membaca transkrip CSV ...")
    dict_transkrip  = baca_transkrip_csv(CSV_PATH)
    semua_transkrip = list(dict_transkrip.values())
    print(f"    Total audio  : {len(dict_transkrip)}")

    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([
        {"Kata": k, "Fonem": " ".join(v)} for k, v in lexicon.items()
    ]).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    print(f"    Vocab        : {len(lexicon)} kata unik  →  HasilLexicon.csv")

    # ── [2] Ekstrak MFCC ───────────────────────────────────────────────────
    print("\n[2] Ekstraksi MFCC ke cache ...")
    cache_mfcc, meta_list = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta_list).to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)
    print(f"    HasilMFCC.csv  ({len(cache_mfcc)} audio di-cache)")

    if not cache_mfcc:
        print("  [ERROR] Tidak ada audio berhasil di-load.")
        return

    X_global = np.vstack(list(cache_mfcc.values()))
    D        = X_global.shape[1]
    print(f"    X_global shape : {X_global.shape}")

    # ── [3] Frame per kata ─────────────────────────────────────────────────
    print("\n[3] Mengumpulkan frame per kata ...")
    frame_per_kata = kumpulkan_frame_per_kata(cache_mfcc, dict_transkrip)

    # ── [4] Training SGMM ─────────────────────────────────────────────────
    print(f"\n[4] Training SGMM ...")
    print(f"    [4a] UBM: I={N_GAUSSIANS}, iter={UBM_ITER} [FIX BUG-4]")
    sgmm = SubspaceGMM(
        n_gaussians  = N_GAUSSIANS,
        subspace_dim = SUBSPACE_DIM,
        ubm_iter     = UBM_ITER,
        sv_iter      = STATE_VEC_ITER,
        sv_lr        = STATE_VEC_LR,
        tol          = TOL,
    )
    sgmm.train_ubm(X_global, verbose=True)

    pd.DataFrame([{
        "I (Gaussians)"       : N_GAUSSIANS,
        "D (Fitur)"           : D,
        "S (Subspace)"        : SUBSPACE_DIM,
        "UBM Iterasi"         : sgmm.ubm.n_iter_done_,
        "UBM Konvergen"       : sgmm.ubm.converged_,
        "UBM LL Akhir"        : round(sgmm.ubm.ll_history_[-1], 6),
        "Shape M_i"           : str(sgmm.M_i.shape),
        "Shape w_i"           : str(sgmm.w_i.shape),
        "Shape Sigma_i"       : str(sgmm.Sigma_i.shape),
    }]).to_csv(f"{DIR_SAVE}/HasilUBM.csv", index=False)
    print(f"    HasilUBM.csv")

    print(f"\n    [4b] State vectors: {len(frame_per_kata)} kata "
          f"[FIX BUG-5, BUG-6, BUG-7]")
    sgmm.fit_state_vectors(frame_per_kata, verbose=True)

    sv_info = []
    for kata, v in sgmm.state_vectors.items():
        sv_info.append({
            "Kata"           : kata,
            "Norm v_j"       : round(float(np.linalg.norm(v)), 4),
            "Mean v_j"       : round(float(v.mean()), 4),
            "Std v_j"        : round(float(v.std()), 4),
            "Jumlah Frame"   : np.vstack(frame_per_kata[kata]).shape[0],
        })
    pd.DataFrame(sv_info).to_csv(
        f"{DIR_SAVE}/HasilStateVectors.csv", index=False
    )
    print(f"    HasilStateVectors.csv  ({len(sv_info)} kata)")

    # ── [5] N-Gram ─────────────────────────────────────────────────────────
    print(f"\n[5] Melatih {N_GRAM}-Gram Language Model ...")
    ngram_model = NGramModel(n=N_GRAM, discount=0.75)
    ngram_model.fit(semua_transkrip)
    print(f"    Vocab={len(ngram_model.vocab)}, "
          f"N-gram count={len(ngram_model.ngram_count)}")

    # ── [6] Dekode + WER (dengan beam search) ─────────────────────────────
    print(f"\n[6] Dekode audio + WER [FIX BUG-1, BUG-3] ...")
    hasil_wer = dekode_semua_audio(
        cache_mfcc, dict_transkrip, sgmm, ngram_model, beam_width=BEAM_WIDTH
    )
    df_wer = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{DIR_SAVE}/HasilWER_SGMM.csv", index=False)

    stat = ringkasan_wer(df_wer, label="SGMM (fixed)")
    pd.DataFrame([stat]).to_csv(
        f"{DIR_SAVE}/HasilRingkasanWER.csv", index=False
    )

    # ── [7] Update M_i ─────────────────────────────────────────────────────
    print("\n[7] Update M_i (1 iterasi) ...")
    sgmm.update_Mi(frame_per_kata, lr=0.001)
    hasil_wer2 = dekode_semua_audio(
        cache_mfcc, dict_transkrip, sgmm, ngram_model,
        beam_width=BEAM_WIDTH, verbose=False,
    )
    df_wer2 = pd.DataFrame(hasil_wer2)
    df_wer2.to_csv(f"{DIR_SAVE}/HasilWER_SGMM_afterMi.csv", index=False)
    ringkasan_wer(df_wer2, label="SGMM + update M_i")

    # ── [8] Hyperparameter Tuning ──────────────────────────────────────────
    if JALANKAN_TUNING:
        print("\n[8] Hyperparameter Tuning SGMM ...")
        hyperparameter_tuning(
            X_global, frame_per_kata, cache_mfcc,
            dict_transkrip, ngram_model, DIR_SAVE,
        )
    else:
        print("\n[8] Tuning dilewati (JALANKAN_TUNING=False)")

    total = time.time() - t_start
    print("\n" + "=" * 60)
    print(f"  SELESAI dalam {total/60:.1f} menit")
    print(f"  Output: {DIR_SAVE}")
    print("=" * 60)


if __name__ == "__main__":
    main()

  SGMM ASR BAHASA JAWA — Pipeline (VERSI PERBAIKAN)
  I=400, S=40, ubm_iter=20
  beam_width=5, lm_weight=0.3

[1] Membaca transkrip CSV ...
    Total audio  : 1749
    Vocab        : 5006 kata unik  →  HasilLexicon.csv

[2] Ekstraksi MFCC ke cache ...
  Mengekstrak MFCC dari 1749 file ...
    [100/1749] cache=100  elapsed=1s  ETA=11s
    [200/1749] cache=200  elapsed=2s  ETA=15s
    [300/1749] cache=300  elapsed=2s  ETA=12s
    [400/1749] cache=400  elapsed=3s  ETA=11s
    [500/1749] cache=500  elapsed=4s  ETA=9s
    [600/1749] cache=600  elapsed=4s  ETA=8s
    [700/1749] cache=700  elapsed=5s  ETA=7s
    [800/1749] cache=800  elapsed=5s  ETA=6s
    [900/1749] cache=900  elapsed=6s  ETA=6s
    [1000/1749] cache=1000  elapsed=7s  ETA=5s
    [1100/1749] cache=1100  elapsed=7s  ETA=4s
    [1200/1749] cache=1200  elapsed=8s  ETA=4s
    [1300/1749] cache=1300  elapsed=8s  ETA=3s
    [1400/1749] cache=1400  elapsed=9s  ETA=2s
    [1500/1749] cache=1500  elapsed=9s  ETA=2s
    [1600/1749] cac

KeyboardInterrupt: 